# Matched-moment Gaussian mixtures and Fourier residual scores

This experiment studies residual score learning with the repository's **`FourierGaussian`, `NoiseProcess`, and `training_loss`**, using a small joint MLP as the backbone.

A fixed Gaussian reference summarizes the data mean and power spectrum. The neural residual models higher-order structure. Gaussian and mixture populations with exactly matched moments let us examine this decomposition using an analytic score.

**Research questions**

1. Can the learned total score improve on the Gaussian reference for a mixture with the same mean and covariance?
2. Do Scalar and Fourier coincide when all frequencies have equal variance?
3. How does spectral heterogeneity affect their relative performance?

**Protocol.** Train with fresh synthetic samples and the shared DSM objective. Evaluate against the exact joint score. Match backbone initialization, data/noise/time streams, optimizer, EMA and update budget across methods.

| Method | Scaled score | Role |
|:--|:--|:--|
| `score` | $h_\theta$ | Direct score DSM |
| `scalar_gaussian` | $\sigma s_{G,\mathrm{scalar}}+b_{\mathrm{scalar}}h_\theta$ | Constant covariance control |
| `fourier_gaussian` | $\sigma s_G+F^{-1}[bFh_\theta]$ | Frequency-dependent Gaussian residual |
| `reference_only` | $\sigma s_G$ | Analytic Gaussian reference |

Start with `smoke` for a short CPU example, `pilot` for protocol development, or `experiment` for repeated training seeds.

![Analytic Gaussian and mixture scores with matched moments](../assets/gaussian_residual.png)

Equal population moments give a common Gaussian reference, while the mixture's exact score contains a residual. Regenerate this illustration with `python scripts/plot_diagnostics.py`.


## Run the notebook

From the repository root:

```bash
uv sync --locked --extra notebooks
uv run --locked --extra notebooks jupyter lab notebooks/gmm_fourier_residual.ipynb
```

Run cells in order. The notebook imports the local checkout and saves results under `saved/gmm_oracle/<preset>/<run_tag>/`. Set `FOURIER_SCORE_ROOT` to select the repository path explicitly.

For a CPU example from a terminal:

```bash
GMM_PRESET=smoke GMM_DEVICE=cpu GMM_RUN_TAG=example \
  uv run --locked --extra notebooks jupyter nbconvert \
  --to notebook --execute notebooks/gmm_fourier_residual.ipynb \
  --ExecutePreprocessor.timeout=600 --output gmm_example.executed.ipynb \
  --output-dir saved/gmm_oracle
```

Choose a new `GMM_RUN_TAG` for each experiment. Reuse the same tag and settings to resume a run or load its completed results. Give concurrent kernels separate tags.

Training supports CPU and CUDA in float32. The exact-score evaluation runs on CPU in float64. Set `GMM_DEVICE=cpu` for CPU training.


In [ ]:
from __future__ import annotations

import os
# Restart the kernel before this cell if CUDA has already been initialized.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import copy
import csv
import hashlib
import html
import json
import math
import platform
import subprocess
import sys
import time
from collections import defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from IPython.display import HTML, display

NOTEBOOK_NAME = "gmm_fourier_residual.ipynb"
NOTEBOOK_VERSION = "2.0.0"
# Set an explicit root only when automatic discovery fails.
REPO_ROOT_OVERRIDE = os.environ.get("FOURIER_SCORE_ROOT")  # e.g. "/workspace/fourier-score"


def find_repo_root() -> Path:
    candidates = ([Path(REPO_ROOT_OVERRIDE).expanduser()] if REPO_ROOT_OVERRIDE else [])
    cwd = Path.cwd().resolve()
    candidates += [cwd, *cwd.parents, cwd / "fourier-score"]
    for p in candidates:
        if (p / "fourier_score/method.py").is_file() and (p / "fourier_score/loss.py").is_file():
            return p.resolve()
    raise FileNotFoundError(
        "Cannot find the fourier-score repository. Place this notebook in its notebooks/ directory "
        "or set FOURIER_SCORE_ROOT. No automatic download is performed."
    )


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fourier_score.method import FourierGaussian, GAUSSIAN_OBJECTIVES
from fourier_score.diffusion import NoiseLevel, NoiseProcess
from fourier_score.loss import training_loss
from fourier_score.spectral import conjugate_symmetrize

# Do not mix a cached import from another checkout into this experiment.
import fourier_score.method as repository_method
assert Path(repository_method.__file__).resolve().parent.parent == ROOT, (
    "Another checkout is already imported. Restart the kernel."
)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def git_output(*args) -> str | None:
    try:
        return subprocess.check_output(
            ["git", "-C", str(ROOT), *args], stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


SOURCE_FILES = [f"fourier_score/{s}.py" for s in ("method", "loss", "diffusion", "spectral")]
SOURCE_HASHES = {p: sha256_file(ROOT / p) for p in SOURCE_FILES}
notebook_candidates = [ROOT / "notebooks" / NOTEBOOK_NAME, Path.cwd() / NOTEBOOK_NAME]
NOTEBOOK_PATH = next((p for p in notebook_candidates if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError(f"Save the notebook as {NOTEBOOK_NAME} to record its source.")


def notebook_code_hash() -> str:
    saved = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    sources = ["".join(c["source"]) for c in saved["cells"] if c["cell_type"] == "code"]
    return hashlib.sha256("\n\n".join(sources).encode()).hexdigest()


# Save before execution. Outputs are excluded from the code hash, so saving them does not invalidate resume.
NOTEBOOK_CODE_SHA = notebook_code_hash()
print("Repository:", ROOT)
print("Python:", platform.python_version(), "| PyTorch:", str(torch.__version__))


## 1. Set the experiment budget

`smoke` runs a quick example, `pilot` supports protocol development, and `experiment` repeats the comparison across training seeds. Select the update budget and learning rate using the validation results, then evaluate the chosen protocol on the held-out test bank.

The full preset has **2 distributions × 3 spectra × 3 seeds × 3 methods = 54 training runs**. The smoke preset uses a 4×4 grid and **12 runs × 80 updates**; pilot and experiment use an 8×8 grid.

Set `GMM_PRESET`, `GMM_DEVICE`, and `GMM_RUN_TAG` before launching the kernel, or edit `CFG` below and save before running.


In [ ]:
@dataclass(frozen=True)
class Config:
    preset: str
    run_tag: str = "v2"
    image_size: int = 8
    rho: float = 0.85
    spectrum_lambdas: tuple[float, ...] = (0.0, 1.0)
    spectrum_knee: float = 0.15
    spectrum_exponent: float = 1.5
    geometry_seed: int = 31415
    distributions: tuple[str, ...] = ("gaussian", "gmm")
    methods: tuple[str, ...] = (
        "score", "scalar_gaussian", "fourier_gaussian"
    )
    seeds: tuple[int, ...] = (42,)
    steps: int = 1500
    eval_every: int = 250
    batch_size: int = 128
    width: int = 128
    depth: int = 3
    time_features: int = 16
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    ema_decay: float = 0.99
    sigma_min: float = 0.10
    sigma_max: float = 3.0
    n_noise_levels: int = 7
    val_per_noise: int = 256
    test_per_noise: int = 1024
    eval_batch_size: int = 128
    frequency_bins: int = 4
    cpu_threads: int = 4
    device: str = "auto"


PRESET = os.environ.get("GMM_PRESET", "smoke")  # "smoke" / "pilot" / "experiment"
PRESETS = {
    "smoke": Config(
        preset="smoke", image_size=4, steps=80, eval_every=40, batch_size=64,
        width=64, depth=2, n_noise_levels=3, val_per_noise=64, test_per_noise=128,
        eval_batch_size=64, frequency_bins=3,
    ),
    "pilot": Config(preset="pilot"),
    "experiment": Config(
        preset="experiment", steps=5000, eval_every=500, width=192,
        seeds=(42, 43, 44), spectrum_lambdas=(0.0, 0.5, 1.0),
        n_noise_levels=9, val_per_noise=512, test_per_noise=2048,
    ),
}
if PRESET not in PRESETS:
    raise ValueError(f"Unknown PRESET: {PRESET}")
CFG = replace(
    PRESETS[PRESET],
    run_tag=os.environ.get("GMM_RUN_TAG", "v2"),
    device=os.environ.get("GMM_DEVICE", "cpu" if PRESET == "smoke" else "auto"),
)
if not CFG.run_tag or Path(CFG.run_tag).name != CFG.run_tag or CFG.run_tag in {".", ".."}:
    raise ValueError("GMM_RUN_TAG must be a single nonempty directory name.")
# If needed, edit once here, save the notebook, and run from the beginning.
# CFG = replace(CFG, run_tag="pilot_lr2e4", learning_rate=2e-4)
# CFG = replace(CFG, run_tag="seed43", seeds=(43,))

assert 2 <= CFG.image_size <= 16
assert 0.0 <= CFG.rho < 1.0
assert CFG.steps >= 1 and CFG.eval_every >= 1 and CFG.batch_size >= 2
assert CFG.sigma_min > 0 and CFG.sigma_max > CFG.sigma_min
assert CFG.n_noise_levels >= 2 and CFG.time_features % 2 == 0
assert CFG.val_per_noise >= 2 and CFG.test_per_noise >= 2
assert all(0 <= x <= 1 for x in CFG.spectrum_lambdas)
assert len(set(CFG.seeds)) == len(CFG.seeds)
assert set(CFG.methods) <= {"score", *GAUSSIAN_OBJECTIVES}
assert set(CFG.distributions) <= {"gaussian", "gmm"}
assert len(set(CFG.methods)) == len(CFG.methods)

DEVICE = torch.device(
    ("cuda" if torch.cuda.is_available() else "cpu") if CFG.device == "auto" else CFG.device
)
if DEVICE.type not in {"cpu", "cuda"}:
    raise ValueError("This notebook supports CPU/CUDA only. On an MPS host, select CPU.")
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable.")

torch.set_num_threads(CFG.cpu_threads)
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

PROCESS_CONFIG = dict(
    type="ve", sigma_min=CFG.sigma_min, sigma_max=CFG.sigma_max,
    num_scales=1000, t_min=0.0, beta_start=1e-4, beta_end=0.02,
)
PROCESS = NoiseProcess(PROCESS_CONFIG)
OUTPUT_ROOT = ROOT / "saved" / "gmm_oracle" / CFG.preset / CFG.run_tag
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def clean_json(x):
    if isinstance(x, dict):
        return {str(k): clean_json(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [clean_json(v) for v in x]
    if isinstance(x, (np.floating, float)):
        return float(x) if math.isfinite(float(x)) else None
    if isinstance(x, np.integer):
        return int(x)
    return x


def atomic_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(clean_json(obj), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    keys = list(dict.fromkeys(k for row in rows for k in row))
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows([{k: clean_json(row.get(k)) for k in keys} for row in rows])
    os.replace(tmp, path)


def show_table(rows, columns=None, max_rows=60):
    if not rows:
        print("No rows.")
        return
    columns = columns or list(rows[0])
    def fmt(v):
        if v is None:
            return "—"
        if isinstance(v, (float, np.floating)):
            return f"{v:.6g}" if np.isfinite(v) else "—"
        return str(v)
    header = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(fmt(r.get(c)))}</td>" for c in columns) + "</tr>" for r in rows[:max_rows])
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    if len(rows) > max_rows:
        print(f"Showing {max_rows}/{len(rows)} rows; complete rows are saved as CSV.")


ENVIRONMENT = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "numpy": str(np.__version__), "platform": platform.platform(),
    "device": str(DEVICE),
    "device_name": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else platform.processor(),
    "cuda_runtime": torch.version.cuda, "cpu_threads": torch.get_num_threads(),
    "deterministic_algorithms": True, "tf32": False,
    "git_commit": git_output("rev-parse", "HEAD"),
    "git_dirty": bool(git_output("status", "--porcelain")),
    "source_sha256": SOURCE_HASHES, "notebook_code_sha256": NOTEBOOK_CODE_SHA,
    "notebook_version": NOTEBOOK_VERSION,
}
PLAN = {"config": asdict(CFG), "environment": ENVIRONMENT, "protocol": "population-statistics / online DSM / final-step test"}
plan_file = OUTPUT_ROOT / "plan.json"
if plan_file.exists() and json.loads(plan_file.read_text()) != clean_json(PLAN):
    raise RuntimeError("This output path contains another configuration/source/runtime. Choose a new GMM_RUN_TAG.")
atomic_json(plan_file, PLAN)
RUN_COUNT = len(CFG.distributions) * len(CFG.spectrum_lambdas) * len(CFG.seeds) * len(CFG.methods)
print(f"Preset={CFG.preset}; device={DEVICE}; {RUN_COUNT} runs × {CFG.steps} updates")
print("Output:", OUTPUT_ROOT)


## 2. Construct distributions with the same mean and covariance

Treat a one-channel $H\times H$ grid as a vector of dimension $d=H^2$. Let $F$ be the orthonormal FFT and let $P$ be a positive, conjugate-symmetric spectrum. Define

$$
C=F^{-1}\operatorname{diag}(P)F,\qquad L=C^{1/2}.
$$

Let $q_j$ denote the rows of a fixed real orthogonal matrix $Q$, interpreted as column vectors in the equations. Set

$$
a_{j,\pm}=\pm\rho\sqrt{d}\,q_j,\qquad
m_{j,\pm}=L a_{j,\pm},\qquad
\Sigma_{\mathrm{within}}=(1-\rho^2)C.
$$

The equally weighted $2d$-component mixture is

$$
p_{\mathrm{GMM}}(x)=\frac{1}{2d}\sum_{j,\pm}
\mathcal N\!\left(x;m_{j,\pm},(1-\rho^2)C\right),
\qquad \mathbb E[X]=0,\qquad \operatorname{Cov}(X)=C.
$$

The Gaussian control is $p_G=\mathcal N(0,C)$. Both populations have the **same full covariance** and use the same Gaussian reference. For $\rho>0$, the mixture has a non-Gaussian joint-score residual.

Keep average power equal to one while varying spectral heterogeneity:

$$
P_k(\lambda)=(1-\lambda)+\lambda P_k^{\mathrm{structured}},
\qquad \frac{1}{d}\sum_k P_k^{\mathrm{structured}}=1.
$$

At $\lambda=0$, Scalar and Fourier define the same operator. Increasing $\lambda$ strengthens the spectral variation. The parameter `rho` controls mixture separation; at `rho=0`, the mixture reduces to the Gaussian control.

All training seeds share the rotation $Q$. Seed repetition measures training variation for a fixed synthetic distribution. To study generalization over mixture geometry, vary `geometry_seed` as a separate experimental axis.

In [ ]:
def fft_filter(x: torch.Tensor, multiplier: torch.Tensor) -> torch.Tensor:
    """Orthonormal full-FFT real filter; valid for conjugate-symmetric multipliers."""
    return torch.fft.ifft2(torch.fft.fft2(x, norm="ortho") * multiplier, norm="ortho").real


def make_power(size: int, lam: float, cfg: Config = CFG) -> torch.Tensor:
    f = torch.fft.fftfreq(size, dtype=torch.float64)
    radius = torch.sqrt(f[:, None].square() + f[None, :].square())
    structured = (1.0 + (radius / cfg.spectrum_knee).square()).pow(-cfg.spectrum_exponent)
    structured = conjugate_symmetrize(structured)
    structured = structured / structured.mean()
    power = (1.0 - lam) * torch.ones_like(structured) + lam * structured
    assert float(power.min()) > 1e-4, "Population power must stay above the adapter floor."
    return power


class MatchedMomentFamily:
    """Small real Gaussian / equal-weight common-covariance GMM, with exact moments.

    Sampling: dense real square root built from the orthonormal Fourier operator.
    Oracle: exact joint score evaluated in real eigencoordinates.
    """
    def __init__(self, cfg: Config, distribution: str, lam: float):
        self.cfg = cfg
        self.distribution = distribution
        self.lam = float(lam)
        self.size = cfg.image_size
        self.d = self.size ** 2
        self.rho = 0.0 if distribution == "gaussian" else cfg.rho
        self.power = make_power(self.size, self.lam, cfg)
        eye_images = torch.eye(self.d, dtype=torch.float64).reshape(self.d, self.size, self.size)
        # Each row is the filtered basis vector; the filter matrix is real symmetric.
        self.root = fft_filter(eye_images, self.power.sqrt()).reshape(self.d, self.d)
        self.covariance = self.root.T @ self.root
        self.within_fraction = 1.0 - self.rho ** 2
        if distribution == "gaussian":
            self.means = torch.zeros(1, self.d, dtype=torch.float64)
        elif distribution == "gmm":
            generator = torch.Generator().manual_seed(cfg.geometry_seed)
            matrix = torch.randn(self.d, self.d, generator=generator, dtype=torch.float64)
            q, r = torch.linalg.qr(matrix)
            # Fix the conventional QR signs for a reproducible geometry.
            q = q * torch.where(torch.diag(r) < 0, -1.0, 1.0)[None, :]
            centers = self.rho * math.sqrt(self.d) * torch.cat((q, -q), dim=0)
            self.means = centers @ self.root
        else:
            raise ValueError(distribution)
        self.n_components = len(self.means)
        self.eigenvalues, self.eigenvectors = torch.linalg.eigh(self.covariance)
        if self.eigenvalues.min() <= 0:
            raise ValueError("Covariance must be positive definite.")
        self.means_eigen = self.means @ self.eigenvectors
        self.stats = {"mean": torch.zeros(1, self.size, self.size), "power": self.power[None].float()}
        self._means32 = self.means.float()
        self._root32 = self.root.float()
        self.case_id = f"{distribution}_lambda{self.lam:g}".replace(".", "p")

    def sample_cpu(self, n: int, generator: torch.Generator, dtype=torch.float32) -> torch.Tensor:
        # One CPU generator isolates training data from network/evaluation randomness.
        index = torch.randint(self.n_components, (n,), generator=generator)
        z = torch.randn(n, self.d, generator=generator, dtype=dtype)
        means = self._means32 if dtype == torch.float32 else self.means.to(dtype)
        root = self._root32 if dtype == torch.float32 else self.root.to(dtype)
        x = means[index] + math.sqrt(self.within_fraction) * (z @ root)
        return x.reshape(n, 1, self.size, self.size)

    def analytic_moments(self):
        mean = self.means.mean(0)
        centered = self.means - mean
        covariance = centered.T @ centered / self.n_components + self.within_fraction * self.covariance
        return mean, covariance

    def log_prob_and_score(self, y: torch.Tensor, alpha: torch.Tensor, sigma: torch.Tensor):
        """Exact JOINT noisy density/score for each (y, alpha, sigma).

        y: [B,1,H,H]; alpha/sigma: [B]. This function is differentiable,
        but is used only to build validation/test banks and test the oracle.
        All mixture responsibilities condition on the ENTIRE y.
        """
        if y.ndim != 4 or y.shape[1:] != (1, self.size, self.size):
            raise ValueError("Expected [B,1,H,H].")
        if alpha.shape != (len(y),) or sigma.shape != (len(y),):
            raise ValueError("alpha and sigma must have shape [B].")
        if bool((sigma <= 0).any()):
            raise ValueError("sigma must be positive.")
        u = self.eigenvectors.to(y)
        eigenvalues = self.eigenvalues.to(y)
        means_eigen = self.means_eigen.to(y)
        y_eigen = y.flatten(1) @ u
        variance = alpha[:, None].square() * self.within_fraction * eigenvalues[None] + sigma[:, None].square()
        delta = y_eigen[:, None, :] - alpha[:, None, None] * means_eigen[None]
        log_components = -0.5 * (
            (delta.square() / variance[:, None, :]).sum(-1)
            + variance.log().sum(-1)[:, None] + self.d * math.log(2.0 * math.pi)
        ) - math.log(self.n_components)
        log_density = torch.logsumexp(log_components, dim=1)
        responsibility = torch.softmax(log_components, dim=1)
        score_eigen = -(responsibility[:, :, None] * delta / variance[:, None, :]).sum(1)
        score = (score_eigen @ u.T).reshape_as(y)
        return log_density, score

    def gaussian_score(self, y, alpha, sigma, scalar=False):
        power = self.power.to(y)
        if scalar:
            power = power.mean().expand_as(power)
        denom = alpha[:, None, None, None].square() * power + sigma[:, None, None, None].square()
        return -fft_filter(y, denom.reciprocal())


FAMILIES = {
    (kind, float(lam)): MatchedMomentFamily(CFG, kind, float(lam))
    for kind in CFG.distributions for lam in CFG.spectrum_lambdas
}
show_table([
    dict(distribution=f.distribution, spectrum_lambda=f.lam, dimension=f.d,
         components=f.n_components, power_mean=float(f.power.mean()),
         power_min=float(f.power.min()), power_max=float(f.power.max()),
         spectral_condition=float(f.power.max() / f.power.min()))
    for f in FAMILIES.values()
])

## 3. Compute the exact noisy score

For the forward marginal $y=\alpha_t x+\sigma_t\epsilon$, let $\gamma_j(y,t)$ be the component responsibility conditioned on the **entire observation**. The exact noisy joint score is

$$
C_{j,t}=\alpha_t^2\Sigma_j+\sigma_t^2I,\qquad
s_*(y,t)=-\sum_j\gamma_j(y,t)C_{j,t}^{-1}(y-\alpha_t m_j).
$$

The score is evaluated in real eigencoordinates, with log-sum-exp for the mixture density. Training uses **VE noise**, with $\alpha=1$.

For Gaussian data, the optimal residual is zero, while individual DSM targets retain stochastic variance. Finite-minibatch optimization measures the effect of this variation around the exact reference.

![Analytic forward and reverse stochastic processes](../assets/stochastic_process.png)

This one-dimensional illustration uses $\sigma^2(t)=4t$ and the exact Gaussian-mixture score. The background shows the analytic density; paths use numerical SDE and ODE integration. Reverse SDE paths start from the exact finite-time mixture, and the ODE trajectories are viewed in both time directions.


In [ ]:
def run_numerical_tests() -> list[dict]:
    checks = []
    def record(name, value=0.0):
        checks.append({"test": name, "status": "PASS", "max_error": float(value)})

    generator = torch.Generator().manual_seed(271828)
    size = min(CFG.image_size, 4)
    small = replace(CFG, image_size=size)
    gaussian = MatchedMomentFamily(small, "gaussian", 1.0)
    mixture = MatchedMomentFamily(small, "gmm", 1.0)
    flat = MatchedMomentFamily(small, "gmm", 0.0)

    for family in (gaussian, mixture, flat):
        mean, covariance = family.analytic_moments()
        err = (covariance - family.covariance).abs().max()
        torch.testing.assert_close(mean, torch.zeros_like(mean), atol=2e-12, rtol=0)
        torch.testing.assert_close(covariance, family.covariance, atol=2e-11, rtol=2e-11)
        torch.testing.assert_close(family.power, conjugate_symmetrize(family.power), atol=1e-14, rtol=0)
        record(f"population moments / {family.case_id}", err)
    torch.testing.assert_close(gaussian.stats["power"], mixture.stats["power"], atol=0, rtol=0)
    record("Gaussian and GMM use identical reference statistics")

    y = torch.randn(4, 1, size, size, generator=generator, dtype=torch.float64)
    alpha = torch.tensor([1.0, 0.8, 0.65, 0.4], dtype=torch.float64)
    sigma = torch.tensor([0.15, 0.4, 0.8, 1.5], dtype=torch.float64)
    y_grad = y.clone().requires_grad_(True)
    log_oracle, score_oracle = mixture.log_prob_and_score(y_grad, alpha, sigma)
    covariance = (alpha[:, None, None].square() * mixture.within_fraction * mixture.covariance
                  + sigma[:, None, None].square() * torch.eye(mixture.d, dtype=torch.float64))
    dense = torch.distributions.MultivariateNormal(
        loc=alpha[:, None, None] * mixture.means[None],
        covariance_matrix=covariance[:, None],
    )
    log_dense = torch.logsumexp(dense.log_prob(y_grad.flatten(1)[:, None]), dim=1) - math.log(mixture.n_components)
    grad_dense, = torch.autograd.grad(log_dense.sum(), y_grad)
    torch.testing.assert_close(log_oracle, log_dense, atol=2e-9, rtol=2e-9)
    torch.testing.assert_close(score_oracle, grad_dense, atol=2e-9, rtol=2e-9)
    record("joint oracle = dense log-density autograd (nontrivial alpha)", (score_oracle - grad_dense).abs().max().detach())

    _, gaussian_oracle = gaussian.log_prob_and_score(y, alpha, sigma)
    ref = FourierGaussian(gaussian.stats, backend="fft").double()
    reference = ref.scaled_score(torch.zeros_like(y), y, alpha, sigma) / sigma[:, None, None, None]
    torch.testing.assert_close(reference, gaussian_oracle, atol=8e-6, rtol=2e-6)
    record("Gaussian oracle = repository Fourier reference", (reference - gaussian_oracle).abs().max())

    raw_f = torch.randn(y.shape, generator=generator, dtype=torch.float64, requires_grad=True)
    raw_s = raw_f.detach().clone().requires_grad_(True)
    rf = FourierGaussian(flat.stats, backend="fft").double()
    rs = FourierGaussian(flat.stats, backend="fft", covariance="scalar").double()
    out_f = rf.scaled_score(raw_f, y, alpha, sigma)
    out_s = rs.scaled_score(raw_s, y, alpha, sigma)
    grad_f, = torch.autograd.grad(out_f.square().sum(), raw_f)
    grad_s, = torch.autograd.grad(out_s.square().sum(), raw_s)
    torch.testing.assert_close(out_f, out_s, atol=2e-12, rtol=2e-12)
    torch.testing.assert_close(grad_f, grad_s, atol=2e-12, rtol=2e-12)
    record("flat spectrum: Scalar/Fourier outputs AND gradients", (out_f - out_s).abs().max().detach())

    e = torch.randn(y.shape, generator=generator, dtype=torch.float64)
    pixel = e.square().flatten(1).mean(1)
    spectral = torch.fft.fft2(e, norm="ortho").abs().square().flatten(1).mean(1)
    torch.testing.assert_close(pixel, spectral, atol=2e-12, rtol=2e-12)
    record("full orthonormal FFT Parseval", (pixel - spectral).abs().max())

    # Analytic target variance for population power p and reference variance q.
    p = mixture.power
    s2 = 0.7 ** 2
    q = torch.full_like(p, float(p.mean()))
    r_q = (s2 * p + q.square()) / (q + s2).square()
    r_p = p / (p + s2)
    gap = s2 * (q - p).square() / ((q + s2).square() * (p + s2))
    torch.testing.assert_close(r_q - r_p, gap, atol=2e-12, rtol=2e-12)
    record("reference covariance mismatch identity", (r_q - r_p - gap).abs().max())
    return checks


NUMERICAL_TESTS = run_numerical_tests()
atomic_json(OUTPUT_ROOT / "numerical_tests.json", NUMERICAL_TESTS)

## 4. Use the repository's DSM objective and a shared backbone

Every learned method uses the same loss:

$$
\mathcal L=\mathbb E\left[\frac{1}{d}
\left\|\sigma s_\theta(y,t)+\epsilon\right\|^2\right].
$$

Training uses the stochastic denoising target. The adapter retains frequency-dependent $b_k^2$ weighting within the shared DSM objective.

A joint MLP observes the full noisy spatial grid. Its last linear layer starts at zero, so each Gaussian method initially equals its corresponding reference score. Step-zero evaluation records the initial total scores.

All methods share Adam, gradient clipping and EMA settings.

In [ ]:
class SmallJointMLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        frequencies = 2.0 ** torch.arange(cfg.time_features // 2, dtype=torch.float32)
        self.register_buffer("frequencies", frequencies, persistent=False)
        layers = []
        n_in = cfg.image_size ** 2 + cfg.time_features
        for _ in range(cfg.depth):
            layers.extend([nn.Linear(n_in, cfg.width), nn.SiLU()])
            n_in = cfg.width
        layers.append(nn.Linear(n_in, cfg.image_size ** 2))
        self.net = nn.Sequential(*layers)
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, y, sigma):
        normalized_time = (sigma.log() - math.log(self.cfg.sigma_min)) / math.log(self.cfg.sigma_max / self.cfg.sigma_min)
        phase = math.pi * normalized_time[:, None] * self.frequencies[None]
        embedding = torch.cat([phase.sin(), phase.cos()], dim=1)
        return self.net(torch.cat([y.flatten(1), embedding], dim=1)).reshape_as(y)


class ToyScoreModel(nn.Module):
    def __init__(self, cfg: Config, stats: dict, method: str):
        super().__init__()
        self.method = method
        self.process = NoiseProcess(PROCESS_CONFIG)
        self.backbone = SmallJointMLP(cfg)
        self.reference = None
        if method in GAUSSIAN_OBJECTIVES:
            self.reference = FourierGaussian(stats, backend="fft", **GAUSSIAN_OBJECTIVES[method])
        elif method != "score":
            raise ValueError(method)

    def scaled_score(self, y, level):
        raw = self.backbone(y, level.sigma)
        if self.reference is None:
            return raw
        return self.reference.scaled_score(raw, y, level.alpha, level.sigma)

    def forward(self, y, level):
        return self.scaled_score(y, level) / level.sigma[:, None, None, None]


def tensor_state_hash(state: dict) -> str:
    h = hashlib.sha256()
    for name, value in sorted(state.items()):
        h.update(name.encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def initial_backbone_state(seed: int):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        backbone = SmallJointMLP(CFG)
    return {k: v.detach().clone() for k, v in backbone.state_dict().items()}


INITIAL_STATES = {seed: initial_backbone_state(seed) for seed in CFG.seeds}
INITIAL_HASHES = {seed: tensor_state_hash(state) for seed, state in INITIAL_STATES.items()}


def make_model(family, method, seed):
    # Backbone initialization uses a seed separate from the data stream.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model = ToyScoreModel(CFG, family.stats, method)
    model.backbone.load_state_dict(INITIAL_STATES[seed], strict=True)
    assert tensor_state_hash(model.backbone.state_dict()) == INITIAL_HASHES[seed]
    return model.to(DEVICE)


# Mean-reduced DSM objective.
_example_family = next(iter(FAMILIES.values()))
_example = make_model(_example_family, "fourier_gaussian", CFG.seeds[0])
_gen = torch.Generator().manual_seed(123)
_x = _example_family.sample_cpu(8, _gen).to(DEVICE)
_lev = PROCESS.sample(8, DEVICE, _gen)
_noise = torch.randn(_x.shape, generator=_gen).to(DEVICE)
_y = PROCESS.perturb(_x, _lev, _noise)
_manual = (_example.scaled_score(_y, _lev) + _noise).square().mean()
_actual = training_loss(_example, _x, _lev, _noise, reduction="mean")
torch.testing.assert_close(_manual, _actual, atol=1e-7, rtol=1e-6)
print("Backbone parameters:", sum(p.numel() for p in _example.backbone.parameters()))
del _example, _x, _lev, _noise, _y, _manual, _actual

## 5. Fixed validation and held-out test banks

The primary metric is **scaled true-score error** against the exact oracle:

$$
E_\theta=\mathbb E_{t,y}\left[
\frac{\sigma_t^2}{d}\left\|s_\theta(y,t)-s_*(y,t)\right\|^2\right].
$$

A midpoint grid in log-$\sigma$ approximates the time integral. The ratio $E_\theta/E_G<1$ indicates a more accurate learned score on the same bank.

Subtracting the common reference gives the same squared error in residual coordinates. Residual cosine similarity provides an additional diagnostic.

The validation bank provides learning curves. The independent test bank evaluates EMA weights at the **predetermined final update**. Shared banks across methods and seeds make the reported standard deviation a measure of training variation conditional on those banks.

In [ ]:
@dataclass
class OracleBank:
    split: str
    entries: list[dict]
    fingerprint: str
    n_observations: int


def bank_seed(family, split: str) -> int:
    payload = f"gmm-oracle-v1:{family.case_id}:{split}:independent-evaluation"
    return 1000000 + int(hashlib.sha256(payload.encode()).hexdigest()[:8], 16)


@torch.no_grad()
def make_bank(family: MatchedMomentFamily, split: str) -> OracleBank:
    n = CFG.val_per_noise if split == "validation" else CFG.test_per_noise
    generator = torch.Generator().manual_seed(bank_seed(family, split))
    entries = []
    digest = hashlib.sha256()
    # Midpoint quadrature in log sigma.
    t_grid = (torch.arange(CFG.n_noise_levels, dtype=torch.float32) + 0.5) / CFG.n_noise_levels
    for noise_bin, t in enumerate(t_grid):
        coordinate = torch.full((n,), float(t))
        level = PROCESS.level(coordinate)
        clean = family.sample_cpu(n, generator)
        noise = torch.randn(clean.shape, generator=generator)
        y = PROCESS.perturb(clean, level, noise)  # input is exactly the FP32 value seen by the model
        oracle_parts, reference_parts, scalar_parts = [], [], []
        for lo in range(0, n, CFG.eval_batch_size):
            hi = min(lo + CFG.eval_batch_size, n)
            yy = y[lo:hi].double()
            aa, ss = level.alpha[lo:hi].double(), level.sigma[lo:hi].double()
            _, exact = family.log_prob_and_score(yy, aa, ss)
            ref = family.gaussian_score(yy, aa, ss)
            scalar = family.gaussian_score(yy, aa, ss, scalar=True)
            oracle_parts.append(ss[:, None, None, None] * exact)
            reference_parts.append(ss[:, None, None, None] * ref)
            scalar_parts.append(ss[:, None, None, None] * scalar)
        entry = {
            "noise_bin": noise_bin, "sigma": float(level.sigma[0]), "coordinate": float(t),
            "y": y, "noise": noise, "oracle_scaled": torch.cat(oracle_parts),
            "reference_scaled": torch.cat(reference_parts), "scalar_reference_scaled": torch.cat(scalar_parts),
        }
        for key in ("y", "noise", "oracle_scaled", "reference_scaled", "scalar_reference_scaled"):
            if not torch.isfinite(entry[key]).all():
                raise FloatingPointError(f"Nonfinite {key} in {family.case_id} / {split}")
            digest.update(entry[key].contiguous().numpy().tobytes())
        digest.update(np.asarray([entry["sigma"], entry["coordinate"]], dtype=np.float64).tobytes())
        entries.append(entry)
    return OracleBank(split, entries, digest.hexdigest(), n * len(entries))


_freq = torch.fft.fftfreq(CFG.image_size, dtype=torch.float64)
_radius = torch.sqrt(_freq[:, None].square() + _freq[None, :].square())
BAND_IDS = (_radius / math.sqrt(0.5) * CFG.frequency_bins).long().clamp_max(CFG.frequency_bins - 1)
BAND_MASKS = [(BAND_IDS == j) for j in range(CFG.frequency_bins)]
BAND_COUNTS = [int(m.sum()) for m in BAND_MASKS]


@torch.no_grad()
def evaluate_model(model: ToyScoreModel | None, family, bank: OracleBank, reference="fourier") -> dict:
    """All reported numerical errors are accumulated on CPU float64.

    model=None evaluates the analytic Gaussian reference.
    """
    if model is not None:
        model.eval()
    rows = []
    total_error = total_ref = total_dsm = total_dot = total_rhat = 0.0
    total_n = 0
    total_band = np.zeros(CFG.frequency_bins, dtype=np.float64)
    for entry in bank.entries:
        predictions = []
        y_all = entry["y"]
        if model is None:
            key = "reference_scaled" if reference == "fourier" else "scalar_reference_scaled"
            predicted = entry[key]
        else:
            for lo in range(0, len(y_all), CFG.eval_batch_size):
                y = y_all[lo:lo + CFG.eval_batch_size].to(DEVICE)
                # Reuse the EXACT stored sigma. Recomputing exp(log_sigma) on CUDA
                # can differ from the CPU bank by an ulp and alter the oracle comparison.
                level = NoiseLevel(
                    alpha=torch.ones(len(y), device=DEVICE),
                    sigma=torch.full((len(y),), entry["sigma"], device=DEVICE),
                    coordinate=torch.full((len(y),), entry["coordinate"], device=DEVICE),
                )
                predictions.append(model.scaled_score(y, level).detach().cpu().double())
            predicted = torch.cat(predictions)
        true = entry["oracle_scaled"]
        error = predicted - true
        rstar = true - entry["reference_scaled"]
        rhat = predicted - entry["reference_scaled"]
        mse = error.square().flatten(1).mean(1)
        reference_error = rstar.square().flatten(1).mean(1)
        noisy_dsm = (predicted + entry["noise"].double()).square().flatten(1).mean(1)
        dot = (rhat * rstar).flatten(1).mean(1)
        rhat_energy = rhat.square().flatten(1).mean(1)
        energies = torch.fft.fft2(error, norm="ortho").abs().square().mean(dim=(0, 1))
        band_values = [float(energies[mask].mean()) if count else None for mask, count in zip(BAND_MASKS, BAND_COUNTS)]
        n = len(mse)
        err_value, ref_value = float(mse.mean()), float(reference_error.mean())
        row = {
            "noise_bin": entry["noise_bin"], "sigma": entry["sigma"], "n": n,
            "score_error": err_value,
            "unweighted_score_error": err_value / entry["sigma"] ** 2,
            "reference_error": ref_value,
            "relative_to_gaussian": err_value / ref_value if ref_value > 1e-10 else None,
            "dsm_pixel_mean": float(noisy_dsm.mean()),
            **{f"frequency_band_{i}": v for i, v in enumerate(band_values)},
        }
        # Check weighting of full-FFT band diagnostics against canonical pixel mean.
        reconstructed = sum((v or 0.0) * c for v, c in zip(band_values, BAND_COUNTS)) / family.d
        if not math.isclose(reconstructed, err_value, rel_tol=2e-8, abs_tol=2e-11):
            raise AssertionError("Frequency-band means do not reproduce pixel score error.")
        rows.append(row)
        total_n += n
        total_error += float(mse.sum())
        total_ref += float(reference_error.sum())
        total_dsm += float(noisy_dsm.sum())
        total_dot += float(dot.sum())
        total_rhat += float(rhat_energy.sum())
        total_band += n * np.asarray([0.0 if v is None else v for v in band_values])
    if not np.isfinite([total_error, total_ref, total_dsm, total_dot, total_rhat]).all():
        raise FloatingPointError("Nonfinite evaluation metric.")
    denominator = math.sqrt(max(total_ref * total_rhat, 0.0))
    return {
        "split": bank.split, "n_observations": total_n, "bank_sha256": bank.fingerprint,
        "score_error": total_error / total_n,
        "reference_error": total_ref / total_n,
        "relative_to_gaussian": total_error / total_ref if total_ref / total_n > 1e-10 else None,
        "residual_cosine": total_dot / denominator if denominator > 1e-10 else None,
        "dsm_pixel_mean": total_dsm / total_n,
        "score_error_by_frequency": [float(total_band[i] / total_n) if BAND_COUNTS[i] else None for i in range(CFG.frequency_bins)],
        "per_noise": rows,
    }


atomic_json(OUTPUT_ROOT / "frequency_bands.json", {
    "modes_per_channel": BAND_COUNTS,
    "edges_cycles_per_pixel": np.linspace(0, math.sqrt(0.5), CFG.frequency_bins + 1).tolist(),
    "definition": "Full orthonormal FFT; DC and both conjugate partners; mode-count weighting.",
})
print("Band mode counts:", BAND_COUNTS, "sum=", sum(BAND_COUNTS))

## 6. Paired training and EMA

Within each `(distribution, lambda, seed)`, all methods receive the **same initial backbone and data/time/noise stream**. Every update uses fresh population samples.

Each run saves `checkpoint.pt` and `metrics.json`. Checkpoints contain the model, EMA, optimizer and training state. An interrupted run continues from its last saved update.

`optimizer_seconds` measures data sampling, training and EMA. `evaluation_seconds` measures evaluation separately. Total session time also includes oracle-bank construction and checkpoint writing.

Run the next cell to train the selected comparisons.


In [ ]:
def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


@torch.no_grad()
def update_ema(ema, model, decay):
    for pe, p in zip(ema.backbone.parameters(), model.backbone.parameters()):
        pe.lerp_(p, 1.0 - decay)


def to_cpu_tree(obj):
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_cpu_tree(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(to_cpu_tree(v) for v in obj)
    return obj


def run_signature(family, method, seed, validation, test):
    description = {
        "config": asdict(CFG), "family": family.case_id, "method": method, "seed": seed,
        "initial_backbone_sha256": INITIAL_HASHES[seed],
        "source_sha256": SOURCE_HASHES, "notebook_code_sha256": NOTEBOOK_CODE_SHA,
        "environment": ENVIRONMENT,
        "validation_sha256": validation.fingerprint, "test_sha256": test.fingerprint,
    }
    return hashlib.sha256(json.dumps(description, sort_keys=True).encode()).hexdigest()


def train_one(family, method, seed, validation, test, *, stop_after=None):
    """Train to CFG.steps and evaluate the final EMA weights.

    stop_after pauses after saving a complete update for resume tests.
    """
    run_dir = OUTPUT_ROOT / family.case_id / f"{method}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = run_dir / "checkpoint.pt"
    signature = run_signature(family, method, seed, validation, test)
    model = make_model(family, method, seed)
    ema = copy.deepcopy(model).eval()
    ema.requires_grad_(False)
    optimizer = torch.optim.Adam(
        model.backbone.parameters(), lr=CFG.learning_rate,
        betas=(0.9, 0.999), eps=1e-8, weight_decay=CFG.weight_decay,
    )
    generator = torch.Generator().manual_seed(100000 + seed)
    state = {
        "signature": signature, "case_id": family.case_id,
        "distribution": family.distribution, "spectrum_lambda": family.lam,
        "method": method, "seed": int(seed), "step": 0, "completed": False,
        "initial_backbone_sha256": INITIAL_HASHES[seed],
        "optimizer_seconds": 0.0, "evaluation_seconds": 0.0,
        "training_stream_first_batch_sha256": None,
        "validation": [], "test": None,
    }

    def save_state():
        payload = {
            "state": state, "model": model.state_dict(), "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(), "data_rng": generator.get_state(),
            "torch_cpu_rng": torch.get_rng_state(),
        }
        tmp = checkpoint.with_name("checkpoint.pt.tmp")
        torch.save(to_cpu_tree(payload), tmp)
        os.replace(tmp, checkpoint)
        atomic_json(run_dir / "metrics.json", state)

    if checkpoint.exists():
        payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
        if payload["state"]["signature"] != signature:
            raise RuntimeError(f"Configuration/source/bank mismatch at {run_dir}; use a new run_tag.")
        state = payload["state"]
        if state["completed"]:
            print(f"SKIP completed {family.case_id} / {method} / seed {seed}")
            return state
        model.load_state_dict(payload["model"], strict=True)
        ema.load_state_dict(payload["ema"], strict=True)
        optimizer.load_state_dict(payload["optimizer"])
        generator.set_state(payload["data_rng"].cpu())
        torch.set_rng_state(payload["torch_cpu_rng"].cpu())
        print(f"RESUME {family.case_id} / {method} / seed {seed} at {state['step']}")
    else:
        start_eval = time.perf_counter()
        metric = evaluate_model(ema, family, validation)
        state["evaluation_seconds"] += time.perf_counter() - start_eval
        metric.update(step=0, optimizer_seconds=0.0, last_training_loss=None)
        state["validation"].append(metric)
        save_state()

    model.train()
    sync_device()
    block_start = time.perf_counter()
    starting_step = state["step"]
    for step in range(starting_step + 1, CFG.steps + 1):
        clean_cpu = family.sample_cpu(CFG.batch_size, generator)
        level = PROCESS.sample(CFG.batch_size, DEVICE, generator)
        noise_cpu = torch.randn(clean_cpu.shape, generator=generator)
        if step == 1:
            stream_hash = hashlib.sha256()
            for tensor in (clean_cpu, level.coordinate.cpu(), noise_cpu):
                stream_hash.update(tensor.contiguous().numpy().tobytes())
            state["training_stream_first_batch_sha256"] = stream_hash.hexdigest()
        clean, noise = clean_cpu.to(DEVICE), noise_cpu.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = training_loss(model, clean, level, noise, reduction="mean")
        loss.backward()
        # This checks finite gradients before optimizer.step. All arms share it.
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.backbone.parameters(), CFG.grad_clip, error_if_nonfinite=True
        )
        optimizer.step()
        update_ema(ema, model, CFG.ema_decay)

        should_record = step % CFG.eval_every == 0 or step == CFG.steps or step == stop_after
        if should_record:
            sync_device()
            state["optimizer_seconds"] += time.perf_counter() - block_start
            state["step"] = step
            start_eval = time.perf_counter()
            metric = evaluate_model(ema, family, validation)
            state["evaluation_seconds"] += time.perf_counter() - start_eval
            metric.update(
                step=step, optimizer_seconds=state["optimizer_seconds"],
                last_training_loss=float(loss.detach().cpu()),
                last_gradient_norm=float(gradient_norm.detach().cpu()),
            )
            state["validation"].append(metric)
            save_state()
            print(
                f"{family.case_id:22s} {method:28s} s{seed} {step:5d}/{CFG.steps} "
                f"true-score={metric['score_error']:.5g} "
                f"train_s={state['optimizer_seconds']:.1f}", flush=True
            )
            if stop_after is not None and step == stop_after and step < CFG.steps:
                return state
            model.train()
            sync_device()
            block_start = time.perf_counter()

    # Evaluate the final EMA checkpoint at CFG.steps.
    start_eval = time.perf_counter()
    state["test"] = evaluate_model(ema, family, test)
    state["evaluation_seconds"] += time.perf_counter() - start_eval
    state["test"].update(step=CFG.steps, weights="EMA")
    state["final_ema_backbone_sha256"] = tensor_state_hash(ema.backbone.state_dict())
    state["completed"] = True
    save_state()
    return state

In [ ]:
# Run all configured distributions, spectra, seeds and methods.
ALL_RESULTS = []
REFERENCES = []
BANK_MANIFESTS = []
experiment_started = time.perf_counter()

for family in FAMILIES.values():
    print(f"\n=== {family.case_id}: preparing independent oracle banks ===", flush=True)
    bank_started = time.perf_counter()
    validation_bank = make_bank(family, "validation")
    test_bank = make_bank(family, "test")
    bank_seconds = time.perf_counter() - bank_started
    BANK_MANIFESTS.append({
        "case_id": family.case_id, "preparation_seconds": bank_seconds,
        "validation_sha256": validation_bank.fingerprint, "test_sha256": test_bank.fingerprint,
        "validation_n": validation_bank.n_observations, "test_n": test_bank.n_observations,
        "validation_seed": bank_seed(family, "validation"), "test_seed": bank_seed(family, "test"),
    })
    for split_bank in (validation_bank, test_bank):
        for reference in ("fourier", "scalar"):
            metric = evaluate_model(None, family, split_bank, reference=reference)
            REFERENCES.append({
                "case_id": family.case_id, "distribution": family.distribution,
                "spectrum_lambda": family.lam, "reference": reference, **metric,
            })

    for seed in CFG.seeds:
        # Rotate execution order deterministically; every arm still receives identical streams.
        methods = list(CFG.methods)
        ordering = np.random.default_rng(seed + 20260922)
        ordering.shuffle(methods)
        paired_states = []
        for method in methods:
            result = train_one(family, method, seed, validation_bank, test_bank)
            if not result["completed"]:
                raise RuntimeError("A partial run cannot enter final results.")
            paired_states.append(result)
            ALL_RESULTS.append(result)
        assert len({r["initial_backbone_sha256"] for r in paired_states}) == 1
        assert len({r["training_stream_first_batch_sha256"] for r in paired_states}) == 1
        # Independent CPU RNGs use identical calls each step; the first batch is also fingerprinted.
    atomic_json(OUTPUT_ROOT / "reference_metrics.json", REFERENCES)
    atomic_json(OUTPUT_ROOT / "oracle_bank_manifests.json", BANK_MANIFESTS)
    atomic_json(OUTPUT_ROOT / "all_results.json", ALL_RESULTS)

SESSION_SECONDS = time.perf_counter() - experiment_started
atomic_json(OUTPUT_ROOT / "session.json", {
    "session_seconds": SESSION_SECONDS, "completed_runs": len(ALL_RESULTS),
    "expected_runs": RUN_COUNT, "note": "Includes bank creation/evaluation/checkpoint I/O; resumed runs may be skipped.",
})
print(f"\nCompleted {len(ALL_RESULTS)}/{RUN_COUNT} runs. This session: {SESSION_SECONDS:.1f} s")
print("Artifacts:", OUTPUT_ROOT)

## 7. Summarize the results

`score_error` is the **final-step held-out test** metric. `relative_to_gaussian < 1` indicates lower error than the Gaussian reference. Ratios with numerically zero reference error are left blank.

`seed_sd` reports sample standard deviation for two or more training seeds. `paired_delta_F_minus_S < 0` indicates lower Fourier error for that paired seed.

The tables include all planned methods, seeds and noise bins. The flat spectrum checks operator equivalence. The Gaussian control measures how finite training affects an initially exact reference.


In [ ]:
FINAL_ROWS = []
NOISE_ROWS = []
CURVE_ROWS = []
for result in ALL_RESULTS:
    common = {k: result[k] for k in ("case_id", "distribution", "spectrum_lambda", "method", "seed")}
    metric = result["test"]
    FINAL_ROWS.append({
        **common, "step": result["step"], "weights": "EMA",
        "score_error": metric["score_error"], "reference_error": metric["reference_error"],
        "relative_to_gaussian": metric["relative_to_gaussian"],
        "residual_cosine": metric["residual_cosine"],
        "dsm_pixel_mean": metric["dsm_pixel_mean"],
        "optimizer_seconds": result["optimizer_seconds"],
        "evaluation_seconds": result["evaluation_seconds"],
        "n_test_observations": metric["n_observations"],
    })
    for row in metric["per_noise"]:
        NOISE_ROWS.append({**common, "split": "test", **row})
    for row in result["validation"]:
        CURVE_ROWS.append({
            **common, "split": "validation", "step": row["step"],
            "score_error": row["score_error"], "relative_to_gaussian": row["relative_to_gaussian"],
            "optimizer_seconds": row["optimizer_seconds"],
            "last_training_loss": row["last_training_loss"],
        })


def summarize(rows, keys, value="score_error"):
    groups = defaultdict(list)
    for row in rows:
        if row.get(value) is not None:
            groups[tuple(row[k] for k in keys)].append(float(row[value]))
    summary = []
    for key, values in sorted(groups.items()):
        values = np.asarray(values)
        summary.append({
            **dict(zip(keys, key)), "metric": value, "n_seeds": len(values),
            "mean": float(values.mean()),
            "seed_sd": float(values.std(ddof=1)) if len(values) > 1 else None,
        })
    return summary


FINAL_SUMMARY = summarize(FINAL_ROWS, ["distribution", "spectrum_lambda", "method"])
paired = defaultdict(dict)
for row in FINAL_ROWS:
    paired[(row["distribution"], row["spectrum_lambda"], row["seed"])][row["method"]] = row["score_error"]
PAIRED_ROWS = []
for (kind, lam, seed), values in sorted(paired.items()):
    if {"fourier_gaussian", "scalar_gaussian"} <= values.keys():
        PAIRED_ROWS.append({
            "distribution": kind, "spectrum_lambda": lam, "seed": seed,
            "paired_delta_F_minus_S": values["fourier_gaussian"] - values["scalar_gaussian"],
        })
PAIRED_SUMMARY = summarize(PAIRED_ROWS, ["distribution", "spectrum_lambda"], value="paired_delta_F_minus_S")
REFERENCE_ROWS = [
    {k: r[k] for k in ("distribution", "spectrum_lambda", "reference", "split", "score_error", "n_observations")}
    for r in REFERENCES
]

write_csv(OUTPUT_ROOT / "final_per_seed.csv", FINAL_ROWS)
write_csv(OUTPUT_ROOT / "final_seed_summary.csv", FINAL_SUMMARY)
write_csv(OUTPUT_ROOT / "paired_F_minus_S.csv", PAIRED_ROWS)
write_csv(OUTPUT_ROOT / "paired_seed_summary.csv", PAIRED_SUMMARY)
write_csv(OUTPUT_ROOT / "validation_curves.csv", CURVE_ROWS)
write_csv(OUTPUT_ROOT / "test_noise_frequency.csv", NOISE_ROWS)
write_csv(OUTPUT_ROOT / "reference_only.csv", REFERENCE_ROWS)

print("Final held-out test — lower true-score error is better")
show_table(FINAL_SUMMARY)
print("Paired Fourier minus Scalar — negative favors Fourier on this metric")
show_table(PAIRED_SUMMARY)
print("Per-seed non-Gaussian residual diagnostics")
show_table([r for r in FINAL_ROWS if r["distribution"] == "gmm"], columns=[
    "spectrum_lambda", "method", "seed", "score_error", "reference_error",
    "relative_to_gaussian", "residual_cosine", "optimizer_seconds",
])

## 8. Export learning curves and spectral diagnostics

Figures are exported as PNG, SVG and PDF. Repeated seeds supply uncertainty bands. Reference curves use the analytic Gaussian score. A $10^{-12}$ display floor supports logarithmic axes; CSV files retain the original values.

Time plots use each seed's measured timestamps. Detailed GMM figures show the largest `lambda` specified in the experiment configuration.


In [ ]:
FIGURE_DIR = OUTPUT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
LABELS = {
    "score": "Score DSM", "scalar_gaussian": "Scalar Gaussian",
    "fourier_gaussian": "Fourier Gaussian",
}
PLOT_FLOOR = 1e-12
FOCUS_LAMBDA = max(CFG.spectrum_lambdas)


def finish_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / (filename + ".png"), dpi=220, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / (filename + ".svg"), bbox_inches="tight")
    fig.savefig(FIGURE_DIR / (filename + ".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(fig)


def grouped_points(rows, xkey, value="score_error"):
    grouped = defaultdict(list)
    for row in rows:
        if row.get(value) is not None:
            grouped[row[xkey]].append(float(row[value]))
    xs = sorted(grouped)
    means = np.asarray([np.mean(grouped[x]) for x in xs])
    sds = np.asarray([np.std(grouped[x], ddof=1) if len(grouped[x]) > 1 else 0.0 for x in xs])
    has_repeats = all(len(grouped[x]) > 1 for x in xs)
    return np.asarray(xs), means, sds, has_repeats


if "gmm" in CFG.distributions:
    focus_curves = [r for r in CURVE_ROWS if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA]
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in focus_curves if r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "step")
        ax.plot(xs, np.maximum(mean, PLOT_FLOOR), marker="o", label=LABELS[method])
        if repeats:
            ax.fill_between(xs, np.maximum(mean - sd, PLOT_FLOOR), np.maximum(mean + sd, PLOT_FLOOR), alpha=0.15)
    ref = next(r for r in REFERENCES if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA and r["split"] == "validation" and r["reference"] == "fourier")
    ax.axhline(max(ref["score_error"], PLOT_FLOOR), linestyle="--", label="Gaussian reference only")
    ax.set(xlabel="Optimizer updates", ylabel="Scaled true-score MSE", yscale="log",
           title=f"Validation learning curves | GMM | lambda={FOCUS_LAMBDA:g} | {CFG.preset}")
    ax.legend(fontsize=8)
    finish_figure(fig, "01_learning_curves")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        for seed in CFG.seeds:
            rows = sorted((r for r in focus_curves if r["method"] == method and r["seed"] == seed), key=lambda r: r["step"])
            ax.plot([r["optimizer_seconds"] for r in rows], [max(r["score_error"], PLOT_FLOOR) for r in rows],
                    marker=".", label=f"{LABELS[method]} / s{seed}")
    ax.set(xlabel="Cumulative data + optimizer + EMA seconds", ylabel="Scaled true-score MSE", yscale="log",
           title="Data sampling, optimization and EMA time")
    ax.legend(fontsize=7)
    finish_figure(fig, "02_learning_vs_time")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in FINAL_ROWS if r["distribution"] == "gmm" and r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "spectrum_lambda")
        ax.errorbar(xs, mean, yerr=sd if repeats else None, marker="o", capsize=3, label=LABELS[method])
    refs = sorted((r for r in REFERENCES if r["distribution"] == "gmm" and r["split"] == "test" and r["reference"] == "fourier"), key=lambda r: r["spectrum_lambda"])
    ax.plot([r["spectrum_lambda"] for r in refs], [r["score_error"] for r in refs], linestyle="--", label="Gaussian reference only")
    ax.set(xlabel="Spectrum interpolation lambda", ylabel="Final held-out scaled true-score MSE",
           title="Spectrum sweep | mean ± training-seed SD when repeated")
    ax.legend(fontsize=8)
    finish_figure(fig, "03_spectrum_sweep")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in NOISE_ROWS if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA and r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "sigma")
        ax.errorbar(xs, mean, yerr=sd if repeats else None, marker="o", capsize=3, label=LABELS[method])
    ax.set(xlabel="Noise sigma", xscale="log", ylabel="Final held-out scaled true-score MSE",
           title="Noise-resolved true-score error")
    ax.legend(fontsize=8)
    finish_figure(fig, "04_noise_resolved_error")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        selected = [r for r in ALL_RESULTS if r["distribution"] == "gmm" and r["spectrum_lambda"] == FOCUS_LAMBDA and r["method"] == method]
        values = np.asarray([[np.nan if v is None else v for v in r["test"]["score_error_by_frequency"]] for r in selected])
        mean = values.mean(0)
        sd = values.std(0, ddof=1) if len(values) > 1 else None
        ax.errorbar(np.arange(CFG.frequency_bins), mean, yerr=sd, marker="o", capsize=3, label=LABELS[method])
    ax.set(xlabel="Radial frequency band (DC included)", ylabel="Mean squared Fourier error per mode",
           title="Final true-score error by frequency")
    ax.set_xticks(np.arange(CFG.frequency_bins))
    ax.legend(fontsize=8)
    finish_figure(fig, "05_frequency_resolved_error")

    fig, ax = plt.subplots(figsize=(8, 4.8))
    for method in CFG.methods:
        rows = [r for r in FINAL_ROWS if r["distribution"] == "gmm" and r["method"] == method]
        xs, mean, sd, repeats = grouped_points(rows, "spectrum_lambda", value="relative_to_gaussian")
        ax.errorbar(xs, mean, yerr=sd if repeats else None, marker="o", capsize=3, label=LABELS[method])
    ax.axhline(1.0, linestyle="--", label="Gaussian reference")
    ax.set(xlabel="Spectrum interpolation lambda", ylabel="Model error / Gaussian-reference error",
           title="Non-Gaussian residual learning | below 1 means improvement")
    ax.legend(fontsize=8)
    finish_figure(fig, "06_beyond_gaussian_reference")
else:
    print("No GMM condition: skipping detailed GMM figures.")

print("Figures:", FIGURE_DIR)

## 9. Residual-target second moments

Under VE noise, the residual target has the following second moment:

$$
T=-\epsilon-\sigma s_G(y),\qquad
\mathbb E\left[\left|F(T)_k\right|^2\right]
=b_k^2=\frac{P_k}{P_k+\sigma^2}.
$$

The identity describes the **individual noisy regression target** using population mean and power. It holds for both Gaussian and non-Gaussian data.

The following cell compares Monte Carlo estimates with the analytic value for the configured spectrum.

In [ ]:
@torch.no_grad()
def target_moment_diagnostic(family, n=20000, sigma=0.7, batch_size=512):
    generator = torch.Generator().manual_seed(909090)
    sum_target_power = torch.zeros_like(family.power)
    seen = 0
    while seen < n:
        b = min(batch_size, n - seen)
        x = family.sample_cpu(b, generator, dtype=torch.float64)
        epsilon = torch.randn(x.shape, generator=generator, dtype=torch.float64)
        y = x + sigma * epsilon
        a = torch.ones(b, dtype=torch.float64)
        s = torch.full((b,), sigma, dtype=torch.float64)
        target = -epsilon - sigma * family.gaussian_score(y, a, s)
        sum_target_power += torch.fft.fft2(target, norm="ortho").abs().square().sum(dim=(0, 1))
        seen += b
    empirical = sum_target_power / n
    analytic = family.power / (family.power + sigma ** 2)
    normalized = empirical / analytic
    return {
        "case_id": family.case_id, "n": n, "sigma": sigma,
        "normalized_target_second_moment_mean": float(normalized.mean()),
        "normalized_target_second_moment_min": float(normalized.min()),
        "normalized_target_second_moment_max": float(normalized.max()),
        "power_relative_L2_error": float((empirical - analytic).norm() / analytic.norm()),
    }


MOMENT_ROWS = [target_moment_diagnostic(f, n=4000 if CFG.preset == "smoke" else 20000) for f in FAMILIES.values()]
show_table(MOMENT_ROWS)
write_csv(OUTPUT_ROOT / "target_moment_diagnostic.csv", MOMENT_ROWS)

## 10. Read the results

**Learning beyond the reference.** Compare the learned total score with the Gaussian reference on GMM data across training seeds. With exact population moments, lower held-out true-score error measures learning of higher-order mixture structure.

**Frequency-dependent covariance.** Compare paired Scalar–Fourier differences across `lambda`. The flat spectrum establishes numerical equivalence, and the structured spectra measure the effect of frequency-dependent covariance.

**Gaussian control.** The analytic reference has zero true-score error. The trained models measure the effect of finite-sample optimization around this solution.

The experiment uses a joint MLP, VE noise, online synthetic data and a fixed mixture geometry. CIFAR-10 and Churches provide the corresponding image and latent-space comparisons.

| Artifact | Contents |
|:--|:--|
| `plan.json` | Experiment settings and runtime |
| `numerical_tests.json` | Moment, score and operator checks |
| `final_per_seed.csv`, `final_seed_summary.csv` | Final held-out true-score errors |
| `paired_F_minus_S.csv`, `paired_seed_summary.csv` | Paired covariance comparison |
| `validation_curves.csv` | Learning curves by update and elapsed time |
| `test_noise_frequency.csv` | Noise and frequency diagnostics |
| `reference_only.csv` | Analytic Gaussian controls |
| `target_moment_diagnostic.csv` | Monte Carlo target moments |
| `figures/*.{png,svg,pdf}` | Exported figures |
| `<case>/<method>_seed*/checkpoint.pt` | Training checkpoints |


## References and implementation

- Repository [method.py](../fourier_score/method.py) and [loss.py](../fourier_score/loss.py): Gaussian reference, residual scaling and DSM.
- [Score-SDE](https://arxiv.org/abs/2011.13456): score-based generative modeling and continuous-time noise processes.
- [EDM](https://arxiv.org/abs/2206.00364): Gaussian and denoiser preconditioning.
- PyTorch [FFT normalization](https://docs.pytorch.org/docs/stable/generated/torch.fft.fft2.html) and [logsumexp](https://docs.pytorch.org/docs/stable/generated/torch.logsumexp.html).
- [Figure generation](../README.md#figures): analytic illustrations and export commands.


## Appendix: numerical checks

Moment, score and operator checks are saved in `numerical_tests.json`.


In [ ]:
show_table(NUMERICAL_TESTS)